# 2. EDA & Preprocessing

Di notebook ini, kita memuat data mentah (`all_data_merged.csv`) yang dikumpulkan dari Notebook 01, mengeksplorasinya sebentar, lalu menjalankan pipeline pembersihan teks (Cleaning & Normalization).

In [1]:
import sys
from pathlib import Path
import pandas as pd

base_dir = Path.cwd().parent
sys.path.insert(0, str(base_dir))

from config.settings import DATA_RAW, DATA_PROCESSED
from preprocessing.text_cleaner import TextCleaningPipeline

pd.set_option('display.max_colwidth', 200)

/home/azril/miniconda3/envs/ml/lib/python3.11/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


## 2.1 Load Raw Data

In [2]:
merged_path = DATA_RAW / "all_data_merged.csv"

if not merged_path.exists():
    print(f"File {merged_path} belum ada. Silakan selesaikan Notebook 01 dulu!")
    # Buat dummy data kecil agar notebook tetap bisa dijalankan untuk demonstrasi
    df = pd.DataFrame({
        "text": [
            "Susu protein ini terlalu manis bgttt dan rasanya agak pahit di akhir. zonk ah.",
            "The texture is so chalky, couldn't finish it! But the flavor is okay.",
            "Evolene rasa coklat lumayan enak, gampang larut di shaker.",
            "Ini produk yg mantap, pengiriman jg cepet dari tokped."
        ],
        "source": ["twitter", "twitter", "shopee", "tokopedia"]
    })
else:
    df = pd.read_csv(merged_path)
    print(f"Loaded {len(df)} rows from {merged_path.name}")

df.head()

Loaded 1356 rows from all_data_merged.csv


,tweet_id,text,created_at,user_name,user_screen_name,user_followers,favorite_count,retweet_count,reply_count,language,query,scraped_at,source
0,2021126757185880478,"@Cryptorealis Om GPT jawab ya, panjang lebar sekalian 👇\n\nKasusnya Om Don\n\n2 hari:\n•❌ ga makan nasi\n•✅ makan lauk pauk + sayur\n•☕ kopi siang masih pakai gula\n\nPutusan sementara:\n\n👉 Diet ...",Tue Feb 10 07:39:04 +0000 2026,NaN,NaN,2499,1,0,1,in,susu protein pahit,2026-05-13T14:20:03.192381,twitter
1,1914600537364033747,"Sekarang tengah hangat isu whey protein yang tipu keputusan lab test. Sepanjang aku guna whey, yang paling aku rasa betul-betul legit adalah daripada Hazim Khalim. Whey dia memang rasa pure—pahit,...",Tue Apr 22 08:41:54 +0000 2025,NaN,NaN,183,1,0,0,in,susu protein pahit,2026-05-13T14:20:12.521806,twitter
2,1902286003911491949,Kenapa kok pakai susu biasa bikin lebih “heavy”? Karena susu biasa protein dan lemaknya lebih rendah.\n\nJadi enggak sepowerful itu dalam mengikat partikel di matcha terutama yang bikin umami dan ...,Wed Mar 19 09:08:20 +0000 2025,NaN,NaN,18456,1,0,0,in,susu protein pahit,2026-05-13T14:20:12.521828,twitter
3,1901284812733678033,@ezabcdefg Ini mah protein dari susu pecah gara gara bromelin deh makanya pahit,Sun Mar 16 14:49:58 +0000 2025,NaN,NaN,837,0,0,1,in,susu protein pahit,2026-05-13T14:20:12.521833,twitter
4,1827909465414525369,Aku dibilang kurusan. 🤣\nWkwkwk hasil abis sakit 2mingguan.\nLidah pahit gk doyan makan. \nRutin minum susu protein &amp; kadang air kelapa. \nCuma turun 2kg tapi pipi jadinya kempes &amp; punya r...,Mon Aug 26 03:22:32 +0000 2024,NaN,NaN,459,0,0,0,in,susu protein pahit,2026-05-13T14:20:12.521837,twitter


## 2.2 Text Cleaning Pipeline

Proses ini akan menjalankan:
1. Penghapusan URL, Emojis, dan HTML.
2. Ekspansi bahasa gaul/slang Indonesia (misal: "bgttt" -> "banget").
3. Deteksi bahasa.
4. Stopword removal (dengan menjaga kosakata sensorik seperti "pahit", "manis", "chalky").

**Note:** Deteksi bahasa (`langdetect`) bisa memakan sedikit waktu untuk data besar.

In [3]:
# Inisialisasi pipeline
cleaner = TextCleaningPipeline(expand_slang=True, detect_language=True)

print("Mulai proses pembersihan teks (mungkin butuh beberapa menit untuk ribuan baris)...")
df_clean = cleaner.process_dataframe(df, text_col="text")

print(f"\nSelesai! Sisa data setelah cleaning: {len(df_clean)} baris.")

Mulai proses pembersihan teks (mungkin butuh beberapa menit untuk ribuan baris)...

Selesai! Sisa data setelah cleaning: 1355 baris.


In [4]:
# Mari lihat perbandingan teks mentah vs teks bersih
df_clean[['text', 'clean_text', 'clean_text_no_stop', 'language']].head(10)

,text,clean_text,clean_text_no_stop,language
0,"@Cryptorealis Om GPT jawab ya, panjang lebar sekalian 👇\n\nKasusnya Om Don\n\n2 hari:\n•❌ ga makan nasi\n•✅ makan lauk pauk + sayur\n•☕ kopi siang masih pakai gula\n\nPutusan sementara:\n\n👉 Diet ...","om gpt jawab ya, panjang lebar sekalian kasusnya om don 2 hari tidak makan nasi makan lauk pauk sayur kopi siang masih pakai gula putusan sementara diet om don sah, tapi belum paripurna. 1 tidak m...","om gpt jawab ya, panjang lebar sekalian kasusnya om don 2 hari tidak makan nasi makan lauk pauk sayur kopi siang masih pakai gula putusan sementara diet om don sah, tapi belum paripurna. 1 tidak m...",id
1,"Sekarang tengah hangat isu whey protein yang tipu keputusan lab test. Sepanjang aku guna whey, yang paling aku rasa betul-betul legit adalah daripada Hazim Khalim. Whey dia memang rasa pure—pahit,...","sekarang tengah hangat isu whey protein yang tipu keputusan lab test. sepanjang aku guna whey, yang paling aku rasa betul-betul legit adalah daripada hazim khalim. whey dia memang rasa purepahit, ...","sekarang tengah hangat isu whey protein tipu keputusan lab test. sepanjang guna whey, paling rasa betul-betul legit daripada hazim khalim. whey memang rasa purepahit, tak sedap, rasa macam susu me...",id
2,Kenapa kok pakai susu biasa bikin lebih “heavy”? Karena susu biasa protein dan lemaknya lebih rendah.\n\nJadi enggak sepowerful itu dalam mengikat partikel di matcha terutama yang bikin umami dan ...,kenapa kok pakai susu biasa bikin lebih heavy? karena susu biasa protein dan lemaknya lebih rendah. jadi enggak sepowerful itu dalam mengikat partikel di matcha terutama yang bikin umami dan pahit...,kenapa kok pakai susu biasa bikin lebih heavy? karena susu biasa protein lemaknya lebih rendah. jadi enggak sepowerful mengikat partikel matcha terutama bikin umami pahit. meanwhile heaviness matc...,id
3,@ezabcdefg Ini mah protein dari susu pecah gara gara bromelin deh makanya pahit,ini mah protein dari susu pecah gara gara bromelin makanya pahit,mah protein susu pecah gara gara bromelin makanya pahit,id
4,Aku dibilang kurusan. 🤣\nWkwkwk hasil abis sakit 2mingguan.\nLidah pahit gk doyan makan. \nRutin minum susu protein &amp; kadang air kelapa. \nCuma turun 2kg tapi pipi jadinya kempes &amp; punya r...,aku dibilang kurusan. hasil abis sakit 2mingguan. lidah pahit tidak doyan makan. rutin minum susu protein amp kadang air kelapa. cuma turun 2kg tapi pipi jadinya kempes amp punya rahang kembali.,dibilang kurusan. hasil abis sakit 2mingguan. lidah pahit tidak doyan makan. rutin minum susu protein amp kadang air kelapa. cuma turun 2kg tapi pipi jadinya kempes amp punya rahang kembali.,id
5,"Udh berobat ke dokter Sp A dibfatmawati\nDisarankan minum susu tinggi protein (neocate junior), rasa susunya pahit\nNamun apa boleh buat\nHarus ngikutin apa kata dokter","sudah berobat ke dokter sp a dibfatmawati disarankan minum susu tinggi protein neocate junior, rasa susunya pahit namun apa boleh buat harus ngikutin apa kata dokter","berobat dokter sp dibfatmawati disarankan minum susu tinggi protein neocate junior, rasa susunya pahit namun apa boleh buat harus ngikutin apa kata dokter",id
6,"Rasa kopinya tidak pahit di lidah dan aman di lambung, lalu dipadukan dgn rasa susu yg menyegarkan. Protein susu yg menyatu dgn kopi menciptakan rasa unik yg menjadi ciri khas cappucino dan latte\...","rasa kopinya tidak pahit di lidah dan aman di lambung, lalu dipadukan dengan rasa susu yang menyegarkan. protein susu yang menyatu dengan kopi menciptakan rasa unik yang menjadi ciri khas cappucin...","rasa kopinya tidak pahit lidah aman lambung, lalu dipadukan rasa susu menyegarkan. protein susu menyatu kopi menciptakan rasa unik menjadi ciri khas cappucino latte tunggu apa lagi, segera beli mi...",id
7,@mashijuseyo pernah terbaca sbb ada benda dlm kiwi tu breaks protein dlm susu tu. so dia akan jdi pahit. something like that la😂tpi boleh je letak madu nk hilangkan pahit,pernah terbaca sbb ada benda dlm kiwi tu break

## 2.3 Save Processed Data

In [5]:
output_path = DATA_PROCESSED / "cleaned_data.csv"
df_clean.to_csv(output_path, index=False)
print(f"Data bersih telah disimpan ke: {output_path}")

Data bersih telah disimpan ke: /home/azril/Personal/Projects/DSAI/NUSFTC/nlp_social_listening/data/processed/cleaned_data.csv
